CNN_Scratch

In [3]:
# ======================================================
# CNN SCRATCH - TRAINING & SAVE (STREAMLIT READY)
# ======================================================

import os
import time
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ======================================================
# DEVICE
# ======================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# ======================================================
# PATH CONFIG
# ======================================================
BASE_DIR = r"D:\alzheimer detection.v1i.folder\processed_dataset"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")

SAVE_DIR = r"D:\alzheimer detection.v1i.folder\Dashboard\src\CNN\model"
os.makedirs(SAVE_DIR, exist_ok=True)

os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_PATH = os.path.join(SAVE_DIR, "cnn_scratch_best.pkl")

# ======================================================
# CONFIG
# ======================================================
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 30
PATIENCE = 5
LR = 1e-3

# ======================================================
# TRANSFORM
# ======================================================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ======================================================
# DATASET & DATALOADER
# ======================================================
train_ds = datasets.ImageFolder(TRAIN_DIR, transform=transform)
val_ds   = datasets.ImageFolder(VAL_DIR, transform=transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

class_names = train_ds.classes
print("Classes:", class_names)

# ======================================================
# CNN MODEL
# ======================================================
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = SimpleCNN(NUM_CLASSES).to(device)
print(model)

# ======================================================
# LOSS & OPTIMIZER
# ======================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# ======================================================
# TRAINING LOOP
# ======================================================
best_val_acc = 0.0
patience_counter = 0

train_losses, val_losses = [], []
train_accs, val_accs = [], []

start_time = time.time()

for epoch in range(EPOCHS):
    # ---------------- TRAIN ----------------
    model.train()
    correct, total, running_loss = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()

    train_acc = correct / total

    # ---------------- VALIDATION ----------------
    model.eval()
    val_correct, val_total, val_loss = 0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += preds.eq(labels).sum().item()

    val_acc = val_correct / val_total

    train_losses.append(running_loss / len(train_loader))
    val_losses.append(val_loss / len(val_loader))
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(
        f"[Epoch {epoch+1}/{EPOCHS}] "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Acc: {val_acc*100:.2f}%"
    )

    # ---------------- SAVE BEST ----------------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0

        artifact = {
            "model_state_dict": model.state_dict(),
            "architecture": "CNN Scratch",
            "num_classes": NUM_CLASSES,
            "class_names": class_names,
            "img_size": IMG_SIZE,
            "normalization": {
                "mean": [0.485, 0.456, 0.406],
                "std": [0.229, 0.224, 0.225]
            },
            "best_val_acc": best_val_acc,
            "epoch": epoch + 1
        }

        joblib.dump(artifact, MODEL_PATH)
        print(f">>> Best model saved: {MODEL_PATH}")

    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(">>> Early stopping triggered!")
            break

total_time = time.time() - start_time
print(f"\nTraining finished in {total_time:.2f} seconds")
print("Best Val Acc:", best_val_acc)


DEVICE: cuda
Classes: ['Mild Impairment', 'Moderate Impairment', 'No Impairment', 'Very Mild Impairment']
SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=100352, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=256, out_features=4, bias=True)
  )
)
[Epoch 1/30] Train Acc: 55.18% | Val Acc: 70.35%
>>> Best model saved: D

Resnet50_LoRA_Fine-Tuning

In [4]:
# ======================================================
# ResNet50 + LoRA Fine-Tuning
# OUTPUT: resnet50_lora_model.pkl ONLY
# ======================================================

import os
import time
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from copy import deepcopy

# ======================================================
# DEVICE
# ======================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# ======================================================
# PATH CONFIG
# ======================================================
BASE_DIR = r"D:\alzheimer detection.v1i.folder\processed_dataset"

MODEL_DIR = r"D:\alzheimer detection.v1i.folder\Dashboard\src\CNN\model"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "resnet50_lora_model.pkl")

# ======================================================
# CONFIG
# ======================================================
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 30
PATIENCE = 5
LR = 2e-4

LORA_R = 8
LORA_ALPHA = 16

# ======================================================
# TRANSFORM
# ======================================================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ======================================================
# DATASET
# ======================================================
train_ds = datasets.ImageFolder(os.path.join(BASE_DIR, "train"), transform=transform)
val_ds   = datasets.ImageFolder(os.path.join(BASE_DIR, "val"), transform=transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

class_names = train_ds.classes
print("Classes:", class_names)

# ======================================================
# LORA LINEAR
# ======================================================
class LoRALinear(nn.Module):
    def __init__(self, linear, r, alpha):
        super().__init__()
        self.linear = linear
        self.scaling = alpha / r

        self.lora_down = nn.Linear(linear.in_features, r, bias=False)
        self.lora_up   = nn.Linear(r, linear.out_features, bias=False)

        # freeze base
        self.linear.weight.requires_grad = False
        if self.linear.bias is not None:
            self.linear.bias.requires_grad = False

    def forward(self, x):
        return self.linear(x) + self.lora_up(self.lora_down(x)) * self.scaling

# ======================================================
# APPLY LORA (SAFE)
# ======================================================
def apply_lora(model, r, alpha):
    model = deepcopy(model)
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            parent = model
            parts = name.split(".")
            for p in parts[:-1]:
                parent = getattr(parent, p)
            setattr(parent, parts[-1], LoRALinear(module, r, alpha))
    return model

# ======================================================
# BUILD MODEL
# ======================================================
base_model = models.resnet50(
    weights=models.ResNet50_Weights.IMAGENET1K_V2
)

# replace FC
in_features = base_model.fc.in_features
base_model.fc = nn.Linear(in_features, NUM_CLASSES)

# apply LoRA
model = apply_lora(base_model, LORA_R, LORA_ALPHA).to(device)

# ======================================================
# LOSS & OPTIMIZER
# ======================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR
)

# ======================================================
# TRAINING LOOP
# ======================================================
best_val_acc = 0.0
patience_counter = 0

start_time = time.time()

for epoch in range(EPOCHS):
    # ---------- TRAIN ----------
    model.train()
    correct, total = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()

    train_acc = correct / total

    # ---------- VALIDATION ----------
    model.eval()
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += preds.eq(labels).sum().item()

    val_acc = val_correct / val_total

    print(
        f"[Epoch {epoch+1}/{EPOCHS}] "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Acc: {val_acc*100:.2f}%"
    )

    # ---------- SAVE BEST ----------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0

        artifact = {
            "model_type": "ResNet50 + LoRA",
            "model_state_dict": model.state_dict(),
            "num_classes": NUM_CLASSES,
            "class_names": class_names,
            "img_size": IMG_SIZE,
            "mean": [0.485, 0.456, 0.406],
            "std": [0.229, 0.224, 0.225],
            "lora": {
                "r": LORA_R,
                "alpha": LORA_ALPHA
            }
        }

        joblib.dump(artifact, MODEL_PATH)
        print(f">>> Best model saved: {MODEL_PATH}")

    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(">>> Early stopping")
            break

print(f"\nTraining finished in {time.time() - start_time:.2f}s")
print("Best Val Acc:", best_val_acc)


DEVICE: cuda
Classes: ['Mild Impairment', 'Moderate Impairment', 'No Impairment', 'Very Mild Impairment']
[Epoch 1/30] Train Acc: 76.00% | Val Acc: 82.18%
>>> Best model saved: D:\alzheimer detection.v1i.folder\Dashboard\src\CNN\model\resnet50_lora_model.pkl
[Epoch 2/30] Train Acc: 90.53% | Val Acc: 83.26%
>>> Best model saved: D:\alzheimer detection.v1i.folder\Dashboard\src\CNN\model\resnet50_lora_model.pkl
[Epoch 3/30] Train Acc: 97.46% | Val Acc: 86.58%
>>> Best model saved: D:\alzheimer detection.v1i.folder\Dashboard\src\CNN\model\resnet50_lora_model.pkl
[Epoch 4/30] Train Acc: 97.73% | Val Acc: 90.53%
>>> Best model saved: D:\alzheimer detection.v1i.folder\Dashboard\src\CNN\model\resnet50_lora_model.pkl
[Epoch 5/30] Train Acc: 98.55% | Val Acc: 93.70%
>>> Best model saved: D:\alzheimer detection.v1i.folder\Dashboard\src\CNN\model\resnet50_lora_model.pkl
[Epoch 6/30] Train Acc: 99.27% | Val Acc: 90.48%
[Epoch 7/30] Train Acc: 97.66% | Val Acc: 92.06%
[Epoch 8/30] Train Acc: 96.91% 

Efficientnet_B0_Baseline

In [6]:
# ======================================================
# EfficientNet-B0 Fine-Tuning (CLASSIFIER ONLY)
# Output: .pkl ONLY (Streamlit Ready)
# ======================================================

import os
import torch
import joblib
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torchvision.models import EfficientNet_B0_Weights
from torch.utils.data import DataLoader

# ======================================================
# DEVICE
# ======================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# ======================================================
# PATH CONFIG (PASTI SESUAI PERMINTAAN)
# ======================================================
BASE_DIR = r"D:\alzheimer detection.v1i.folder\processed_dataset"

MODEL_DIR = r"D:\alzheimer detection.v1i.folder\Dashboard\src\CNN\model"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "efficientnet_b0_finetune.pkl"
)

# ======================================================
# CONFIG
# ======================================================
NUM_CLASSES = 4
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15
PATIENCE = 3
LR = 1e-4

# ======================================================
# TRANSFORMS
# ======================================================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ======================================================
# DATASET & DATALOADER
# ======================================================
train_ds = datasets.ImageFolder(
    os.path.join(BASE_DIR, "train"),
    transform=transform
)
val_ds = datasets.ImageFolder(
    os.path.join(BASE_DIR, "test"),
    transform=transform
)

class_names = train_ds.classes
print("Classes:", class_names)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False
)

# ======================================================
# MODEL: EfficientNet-B0 (FINE-TUNE CLASSIFIER ONLY)
# ======================================================
weights = EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)

# ---- FREEZE BACKBONE ----
for param in model.features.parameters():
    param.requires_grad = False

# ---- REPLACE CLASSIFIER ----
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

model = model.to(device)

# ======================================================
# LOSS & OPTIMIZER (CLASSIFIER ONLY)
# ======================================================
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=LR
)

# ======================================================
# TRAINING LOOP
# ======================================================
best_val_acc = 0.0
patience_counter = 0

for epoch in range(EPOCHS):
    # ---------------- TRAIN ----------------
    model.train()
    correct, total, train_loss = 0, 0, 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()

    train_acc = correct / total

    # ---------------- VALIDATION ----------------
    model.eval()
    val_correct, val_total, val_loss = 0, 0, 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += preds.eq(labels).sum().item()

    val_acc = val_correct / val_total

    print(
        f"[Epoch {epoch+1}/{EPOCHS}] "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Acc: {val_acc*100:.2f}%"
    )

    # ---------------- SAVE BEST (.PKL ONLY) ----------------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0

        artifact = {
            "model_state_dict": model.state_dict(),
            "architecture": "EfficientNet-B0 (Fine-Tune Classifier)",
            "num_classes": NUM_CLASSES,
            "class_names": class_names,
            "img_size": IMG_SIZE,
            "normalization": {
                "mean": [0.485, 0.456, 0.406],
                "std": [0.229, 0.224, 0.225]
            },
            "best_val_acc": best_val_acc
        }

        joblib.dump(artifact, MODEL_PATH)
        print(f">>> Best model saved: {MODEL_PATH}")

    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(">>> Early stopping triggered")
            break

print("\n[SUCCESS]")
print("Best Val Acc:", best_val_acc)
print("Model saved at:", MODEL_PATH)


DEVICE: cuda
Classes: ['Mild Impairment', 'Moderate Impairment', 'No Impairment', 'Very Mild Impairment']
[Epoch 1/15] Train Acc: 47.43% | Val Acc: 65.56%
>>> Best model saved: D:\alzheimer detection.v1i.folder\Dashboard\src\CNN\model\efficientnet_b0_finetune.pkl
[Epoch 2/15] Train Acc: 64.36% | Val Acc: 67.81%
>>> Best model saved: D:\alzheimer detection.v1i.folder\Dashboard\src\CNN\model\efficientnet_b0_finetune.pkl
[Epoch 3/15] Train Acc: 67.43% | Val Acc: 67.96%
>>> Best model saved: D:\alzheimer detection.v1i.folder\Dashboard\src\CNN\model\efficientnet_b0_finetune.pkl
[Epoch 4/15] Train Acc: 67.61% | Val Acc: 69.04%
>>> Best model saved: D:\alzheimer detection.v1i.folder\Dashboard\src\CNN\model\efficientnet_b0_finetune.pkl
[Epoch 5/15] Train Acc: 68.94% | Val Acc: 69.34%
>>> Best model saved: D:\alzheimer detection.v1i.folder\Dashboard\src\CNN\model\efficientnet_b0_finetune.pkl
[Epoch 6/15] Train Acc: 70.01% | Val Acc: 70.93%
>>> Best model saved: D:\alzheimer detection.v1i.folder